# 01 — Story Generation

Loops over all schedules × prompts and saves results to `results/`.

In [1]:
import sys, os

import json
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

import config
from src.schedules import get_schedule, get_all_schedules
from src.generation import run_generation

ModuleNotFoundError: No module named 'transformers'

In [ ]:
# ── Load prompts ──────────────────────────────────────────────────────────────
with open(config.DATA_PATH) as f:
    all_prompts = [json.loads(line) for line in f if line.strip()]

prompts = all_prompts[: config.N_PROMPTS]
print(f"Using {len(prompts)} prompts.")
print("Example:", prompts[0]["prompt"][:120])

In [ ]:
# ── Load model & tokenizer ────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    config.MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
)
if device == "cpu":
    model = model.to(device)
model.eval()
print("Model loaded:", config.MODEL_NAME)

In [ ]:
# ── Run generation for each schedule ─────────────────────────────────────────
schedule_map = get_all_schedules(config.N_CHUNKS)

for sched_name in config.SCHEDULES:
    schedule = schedule_map[sched_name]
    out_dir = config.results_dir(config.MODEL_NAME, sched_name)
    print(f"\n=== Schedule: {sched_name} | temps: {schedule} ===")

    run_generation(
        model=model,
        tokenizer=tokenizer,
        prompts=prompts,
        schedule_name=sched_name,
        schedule=schedule,
        tokens_per_chunk=config.TOKENS_PER_CHUNK,
        model_name=config.MODEL_NAME,
        output_dir=out_dir,
    )

print("\nGeneration complete.")